# Class 24: Scipy: Interpolation, Differentiation, and Integration
## Objective: Understand scipy tools appropriate to these numerical methods

These exercises are based on those in "Lecture 16: SciPy: Interpolation, Differentiation and Integration" by Yuan-Sen Ting and available from https://tingyuansen.github.io/coding_essential_for_astronomers/lectures/lecture16-scipy-interpolation-differentiation-integration.html

In [ ]:
import numpy as np
import scipy
import matplotlib.pyplot as plt

# Plotting defaults
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## Section 1: Interpolation (Filling the Gaps)
In astronomy, we often have "sparse" data—measurements taken at irregular intervals because of weather, telescope scheduling, or daylight. Interpolation allows us to estimate the value of a property (like the brightness of a star) at a time when we weren't actually looking.

### Linear Interpolation
Linear interpolation is the simplest form of filling gaps between data points. It assumes that the change between two known values is constant, effectively "connecting the dots" with straight line segments.

**The Equation:**
Given two points $(x_0, y_0)$ and $(x_1, y_1)$, the value $y$ for any $x$ in the interval $[x_0, x_1]$ is calculated as:

$$y = y_0 + (x - x_0) \frac{y_1 - y_0}{x_1 - x_0}$$

**Key Characteristics:**
* **Simplicity:** Fast to compute and robust against "overshooting" data.
* **Continuity:** The resulting curve is continuous ($C^0$ continuity), but its slope (derivative) changes abruptly at every data point, leading to a "jagged" appearance.
* **Astronomy Use Case:** Useful for estimating values in strictly linear relationships or when the data is sampled so densely that the curvature between points is negligible.

### Cubic Spline Interpolation
Cubic spline interpolation fits a series of unique third-degree polynomials between each pair of data points. Unlike linear interpolation, it ensures that the transitions at the data points are perfectly smooth.

**The Equation:**
For each interval between $x_i$ and $x_{i+1}$, the interpolant is a cubic function:

$$S_i(x) = a_i + b_i(x - x_i) + c_i(x - x_i)^2 + d_i(x - x_i)^3$$

To solve for the coefficients $a_i, b_i, c_i,$ and $d_i$, the following constraints are applied to ensure a smooth "spline":
1. **Interpolation:** The curve must pass through all data points.
2. **Continuity:** The segments must meet at the points: $S_i(x_{i+1}) = S_{i+1}(x_{i+1})$.
3. **Smoothness (Slope):** The first derivatives must be equal at the points: $S'_i(x_{i+1}) = S'_{i+1}(x_{i+1})$.
4. **Smoothness (Curvature):** The second derivatives must be equal at the points: $S''_i(x_{i+1}) = S''_{i+1}(x_{i+1})$.

**Key Characteristics:**
* **Visual Appeal:** Produces a "smooth" curve that is often more physically realistic for natural phenomena like stellar orbits or light curves.
* **$C^2$ Continuity:** Both the slope and the rate of change of the slope are continuous.
* **Astronomy Use Case:** The standard for modeling smooth physical processes, such as the radial velocity curve of a star being tugged by an exoplanet.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

# 1. Create sparse "observed" data (e.g., a pulsing star)
x_obs = np.linspace(0, 10, 10)
y_obs = np.sin(x_obs)

# 2. Create the interpolation functions
f_linear = interp1d(x_obs, y_obs, kind='linear')
f_cubic = interp1d(x_obs, y_obs, kind='cubic')

# 3. Predict values on a much denser grid
x_dense = np.linspace(0, 10, 100)
y_linear = f_linear(x_dense)
y_cubic = f_cubic(x_dense)

# Plotting
plt.figure(figsize=(8, 4))
plt.plot(x_obs, y_obs, 'ko', label='Observed Data')
plt.plot(x_dense, y_linear, '--', label='Linear Interpolation')
plt.plot(x_dense, y_cubic, '-', label='Cubic Spline')
plt.legend()
plt.title("Stellar Light Curve Interpolation")
plt.show()

**Test your understanding:** Create a new cubic spline interpolation for a set of observations where x = [0, 2, 4, 6, 8, 10] and y = [1, 5, 2, 8, 4, 10]. Use your interpolator to find the value of y at x = 5.5.

In [ ]:
x_data = [0, 2, 4, 6, 8, 10]
y_data = [1, 5, 2, 8, 4, 10]
# Enter your code here 

## Section 2: Numerical Differentiation and Finding Extrema

Differentiation measures the **instantaneous rate of change** of a function. In astronomy, this is a fundamental tool for converting observed positions into physical dynamics. For example, if we have the measured position of a star $s$ over a period of time $t$, the first derivative $ds/dt$ tells us the star's velocity.

### The Mathematical Definition (The Limit)
Mathematically, the derivative of a function $f(x)$ is defined as the limit of the average rate of change as the interval $h$ approaches zero:

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

While this works for continuous mathematical formulas, astronomical data consists of discrete observations (individual points), meaning we cannot actually take the limit to zero. Instead, we use **Finite Differences**.

### The Finite Difference Approach
When working with arrays of data, we approximate the derivative by calculating the slope between neighboring data points. 

1. **Forward Difference:** Uses the current point and the next point.
   $$f'(x_i) \approx \frac{f(x_{i+1}) - f(x_i)}{h}$$

2. **Central Difference:** Uses the points before and after the current position. This is generally more accurate for astronomical signals as it centers the slope on the point of interest.
   $$f'(x_i) \approx \frac{f(x_{i+1}) - f(x_{i-1})}{2h}$$



### Finding Extrema (Maxima and Minima)
A primary goal in data analysis is finding where a function reaches its peak (Maximum) or its lowest point (Minimum). At these specific points, the rate of change is zero:

$$f'(x) = 0$$

By calculating the numerical derivative of a light curve or a radial velocity plot, we can pinpoint the exact moment an event occurs—such as the "mid-transit" time when an exoplanet is directly in front of its host star.

**Key Characteristics:**
* **Sensitivity to Noise:** Differentiation acts as a "high-pass filter." If your data has small "wiggles" due to sensor noise, the derivative will amplify those wiggles significantly.
* **Step Size:** The accuracy of the approximation depends on $h$ (the spacing between your observations).
* **Python Tool:** While we can calculate these manually, we typically use `numpy.gradient()` for arrays or SciPy's signal processing tools to find peaks in noisy data.

In [ ]:
# We'll use the cubic interpolation from the previous section
y_dense = f_cubic(x_dense)

# 1. Calculate the numerical derivative (the slope)
# numpy.gradient calculates the difference between adjacent points
dy_dx = np.gradient(y_dense, x_dense)

# 2. Find where the derivative is approximately zero (Extrema)
# We can look for where the slope changes from positive to negative
peaks = np.where(np.diff(np.sign(dy_dx)) < 0)[0]

plt.figure(figsize=(8, 4))
plt.plot(x_dense, y_dense, label="Signal")
plt.plot(x_dense, dy_dx, '--', label="Derivative (Slope)")
plt.plot(x_dense[peaks], y_dense[peaks], 'rs', label="Detected Peak")
plt.axhline(0, color='black', lw=0.5)
plt.legend()
plt.show()

**Test your understanding:** Using the x_dense and y_dense variables from the code above, find the Minimum of the curve. (Hint: A minimum occurs where the slope changes from negative to positive, so np.diff(np.sign(dy_dx)) > 0). Print the x-coordinate of the minimum.

In [ ]:
# Enter your code here 

## Section 3: Numerical Integration
Numerical integration is the process of calculating the "total" of a quantity by finding the area under its curve. In astronomy, this is essential for calculating **Total Flux** from a spectrum (energy vs. wavelength) or determining the **Total Distance** traveled by an object if you only have its velocity measurements over time.

The goal is to approximate the definite integral:

$$I = \int_{a}^{b} f(x) \, dx$$

Since astronomical data consists of discrete measurements at specific points $x_i$ with a spacing of $h$, we use weighted sums of these points to estimate the area.

### The Trapezoidal Rule
The Trapezoidal Rule works by approximating the area under the curve as a series of trapezoids connecting each data point. It assumes the function is linear between each observation.

For a single interval between $x_i$ and $x_{i+1}$, the area is:
$$\text{Area}_i \approx \frac{h}{2} [f(x_i) + f(x_{i+1})]$$



When applied across the entire range from $a$ to $b$ with $n$ equally spaced intervals of width $h$, the formula becomes:

$$\int_{a}^{b} f(x) \, dx \approx \frac{h}{2} \left[ f(x_0) + 2f(x_1) + 2f(x_2) + \dots + 2f(x_{n-1}) + f(x_n) \right]$$

* **Pros:** Very robust and works well even for data with sharp changes or high noise.
* **Cons:** Less accurate than higher-order methods for very smooth functions.

### Simpson's Rule
Simpson's Rule provides a more accurate approximation by fitting a **parabola** (a quadratic curve) through sets of three consecutive points. This better accounts for the natural "curvature" in physical data.

For this rule to work, you must have an **even number of intervals** (an odd number of data points). The approximation for the total area is:

$$\int_{a}^{b} f(x) \, dx \approx \frac{h}{3} \left[ f(x_0) + 4f(x_1) + 2f(x_2) + 4f(x_3) + 2f(x_4) + \dots + 4f(x_{n-1}) + f(x_n) \right]$$



* **Pros:** Significantly more accurate for smooth, continuous signals (like a high-resolution stellar spectrum).
* **Cons:** Requires evenly spaced data points and can behave poorly if the data is very "noisy" or has sudden discontinuities.

### Summary of SciPy Tools
There are implementations of both tools in the `scipy.integrate` module:
* `trapezoid(y, x)`: Use this when your data might be noisy or intervals are irregular.
* `simpson(y, x)`: Use this for smooth, precisely measured signals where you want the highest possible accuracy.

In [ ]:
from scipy.interpolate import interp1d

# 1. Define a smooth function and some sparse data points
def func(x):
    return np.cos(x) + 1.5

x_fine = np.linspace(0, np.pi, 200)
y_fine = func(x_fine)

# Use 5 points (4 intervals) - Simpson's needs an odd number of points
x_data = np.linspace(0, np.pi, 5)
y_data = func(x_data)

fig, ax = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# --- Subplot 1: Trapezoidal Rule ---
ax[0].plot(x_fine, y_fine, 'k--', alpha=0.5, label='Actual Function')
ax[0].plot(x_data, y_data, 'ko', label='Observations')
# Fill the trapezoids
for i in range(len(x_data)-1):
    xs = [x_data[i], x_data[i+1]]
    ys = [y_data[i], y_data[i+1]]
    ax[0].fill_between(xs, ys, color='skyblue', alpha=0.4, edgecolor='blue', linewidth=2)
ax[0].set_title("Trapezoidal Rule\n(Linear Approximation)", fontsize=14)
ax[0].set_xlabel("x", fontsize=12)
ax[0].set_ylabel("f(x)", fontsize=12)

# --- Subplot 2: Simpson's Rule ---
ax[1].plot(x_fine, y_fine, 'k--', alpha=0.5, label='Actual Function')
ax[1].plot(x_data, y_data, 'ko', label='Observations')

# Simpson's fits a parabola to every set of 3 points
for i in range(0, len(x_data)-2, 2):
    # Select 3 points
    x_trip = x_data[i:i+3]
    y_trip = y_data[i:i+3]
    # Fit a quadratic (parabola)
    poly = np.polyfit(x_trip, y_trip, 2)
    p = np.poly1d(poly)
    # Generate fine points for the parabola curve
    x_range = np.linspace(x_trip[0], x_trip[-1], 50)
    y_range = p(x_range)
    ax[1].fill_between(x_range, y_range, color='salmon', alpha=0.4, edgecolor='red', linewidth=2)

ax[1].set_title("Simpson's Rule\n(Parabolic Approximation)", fontsize=14)
ax[1].set_xlabel("x", fontsize=12)

# Add a common legend
ax[0].legend(loc='upper right')
plt.tight_layout()
plt.show()

### Key points to observe about the two-panel plot above. 

**Underestimation vs. Overestimation:** In the Trapezoidal plot, there are clear gaps (white space) between the blue trapezoid edges and the dashed function line. This shows where the rule is underestimating the area.

**The "Fit":** In the Simpson's plot, note how the red parabolic edges "hug" the dashed line much more closely. This visually explains why Simpson's rule is generally more accurate for smooth, continuous physical signals.

**Data Requirements:** Simpson's rule requires a specific "triplet" of points to define each parabola, which is why the code loops with a step of 2 (`range(0, len(x_data)-2, 2)`).

### Flux Measurement with Numerical Integration

The next plot shows how we can use these tools to integrate a spectral lien to determine the total flux. 

In [ ]:
from scipy.integrate import trapezoid, simpson

# 1. Create a "Spectral Line" (a Gaussian shape)
x_spec = np.linspace(-5, 5, 100)
y_spec = np.exp(-x_spec**2) # A bell curve

# 2. Calculate the total area (Flux) using both methods
area_trap = trapezoid(y_spec, x_spec)
area_simp = simpson(y_spec, x_spec)

print(f"Total Flux (Trapezoid): {area_trap:.4f}")
print(f"Total Flux (Simpson): {area_simp:.4f}")

plt.fill_between(x_spec, y_spec, color='skyblue', alpha=0.4)
plt.plot(x_spec, y_spec, color='blue')
plt.title("Integrating a Spectral Line")
plt.show()

**Test your understanding:** Suppose you have a constant flux of y = 10 (a flat line) measured from x = 0 to x = 10. Use the trapezoid function to calculate the area. Does the result match the geometric area of a rectangle (10×10)?

In [ ]:
x_test = np.linspace(0, 10, 100)
y_test = np.ones(100) * 10
# Your code here

## Solutions to In-Class Exercises

### Section 1 Solution

In [ ]:
x_data = [0, 2, 4, 6, 8, 10]
y_data = [1, 5, 2, 8, 4, 10]
f = interp1d(x_data, y_data, kind='cubic')
print(f(5.5))

### Section 2 Solution

In [ ]:
valleys = np.where(np.diff(np.sign(dy_dx)) > 0)[0]
print(f"Minimum x-location: {x_dense[valleys][0]}")

### Section 3 Solution

In [ ]:
result = trapezoid(y_test, x_test)
print(result) # Should be 100.0